In [1]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import io
import json
import warnings
warnings.filterwarnings('ignore')

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA
from scipy import stats

# Load FX data
def query_athena(query, database='fx_rates_db', region='us-east-2'):
    athena = boto3.client('athena', region_name=region)
    s3 = boto3.client('s3', region_name=region)
    output_location = 's3://fx-rates-ninpar/athena-results/'
    
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location}
    )
    query_id = response['QueryExecutionId']
    
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)
        state = status['QueryExecution']['Status']['State']
        if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
        time.sleep(1)
    
    if state != 'SUCCEEDED':
        raise Exception("Query failed")
    
    result_location = status['QueryExecution']['ResultConfiguration']['OutputLocation']
    bucket = result_location.split('/')[2]
    key = '/'.join(result_location.split('/')[3:])
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

df_fx = query_athena("""
    SELECT date, rates.inr AS usd_inr, rates.eur AS usd_eur, 
           rates.gbp AS usd_gbp, rates.mxn AS usd_mxn, rates.php AS usd_php
    FROM fx_rates_db.usd
    ORDER BY date ASC
""")
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx = df_fx.set_index('date').sort_index()

# Load Chronos results for comparison
with open('chronos_results.json', 'r') as f:
    chronos_results = json.load(f)

print(f"FX data: {len(df_fx)} days")

FX data: 281 days


In [2]:
def walk_forward_statsforecast(series, model_name='ETS', test_days=60):
    """
    Walk-forward evaluation using StatsForecast.
    Refits model each day on growing training window.
    """
    series = series.dropna()
    if len(series) < test_days + 60:
        return None
    
    results = []
    
    for i in range(test_days, 0, -1):
        train = series[:-i]
        actual = series.iloc[-i]
        naive = train.iloc[-1]
        
        # StatsForecast expects specific dataframe format
        sf_df = pd.DataFrame({
            'unique_id': 'series_1',
            'ds': train.index,
            'y': train.values
        })
        
        # Select model
        if model_name == 'ETS':
            models = [AutoETS(season_length=1)]  # no seasonality for FX
        elif model_name == 'ARIMA':
            models = [AutoARIMA(season_length=1)]
        
        sf = StatsForecast(models=models, freq='B', n_jobs=1, verbose=False)
        
        try:
            sf.fit(df=sf_df)
            forecast = sf.predict(h=1)
            
            # Get prediction (column name varies by model)
            pred_col = model_name if model_name in forecast.columns else forecast.columns[-1]
            forecast_value = forecast[pred_col].iloc[0]
            
            results.append({
                'date': series.index[-i],
                'actual': actual,
                'naive': naive,
                'forecast': forecast_value,
                'naive_error': abs(actual - naive),
                'forecast_error': abs(actual - forecast_value),
                'actual_direction': 1 if actual > naive else 0,
                'forecast_direction': 1 if forecast_value > naive else 0
            })
        except Exception as e:
            continue
    
    return pd.DataFrame(results)

print("Walk-forward function ready")

Walk-forward function ready


In [3]:
print("Running ETS evaluation (60 test days × 5 pairs)...")
print("This is fast — StatsForecast fits in seconds per day\n")

ets_evaluation = {}
arima_evaluation = {}

for col in ['usd_inr', 'usd_eur', 'usd_gbp', 'usd_mxn', 'usd_php']:
    print(f"  {col.upper()} — ETS...", end=" ", flush=True)
    result_ets = walk_forward_statsforecast(df_fx[col], 'ETS', test_days=60)
    if result_ets is not None:
        ets_evaluation[col] = result_ets
        acc = (result_ets['actual_direction'] == result_ets['forecast_direction']).mean() * 100
        print(f"done — direction: {acc:.1f}%", end=" | ")
    
    print(f"ARIMA...", end=" ", flush=True)
    result_arima = walk_forward_statsforecast(df_fx[col], 'ARIMA', test_days=60)
    if result_arima is not None:
        arima_evaluation[col] = result_arima
        acc = (result_arima['actual_direction'] == result_arima['forecast_direction']).mean() * 100
        print(f"done — direction: {acc:.1f}%")

print("\nClassical baselines complete")

Running ETS evaluation (60 test days × 5 pairs)...
This is fast — StatsForecast fits in seconds per day

  USD_INR — ETS... done — direction: 53.3% | ARIMA... done — direction: 61.7%
  USD_EUR — ETS... done — direction: 61.7% | ARIMA... done — direction: 46.7%
  USD_GBP — ETS... done — direction: 58.3% | ARIMA... done — direction: 63.3%
  USD_MXN — ETS... done — direction: 60.0% | ARIMA... done — direction: 63.3%
  USD_PHP — ETS... done — direction: 55.0% | ARIMA... done — direction: 56.7%

Classical baselines complete


In [4]:
print("=== Classical Baselines vs Chronos ===\n")

all_results = []

for col in ['usd_inr', 'usd_eur', 'usd_gbp', 'usd_mxn', 'usd_php']:
    # ETS
    if col in ets_evaluation:
        r = ets_evaluation[col]
        ets_rmse = (r['forecast_error'] ** 2).mean() ** 0.5
        naive_rmse = (r['naive_error'] ** 2).mean() ** 0.5
        ets_theil = ets_rmse / naive_rmse
        ets_dir = (r['actual_direction'] == r['forecast_direction']).mean() * 100
        all_results.append({'pair': col.upper(), 'model': 'ETS', 'theil_u': ets_theil, 'direction_acc': ets_dir})
    
    # ARIMA
    if col in arima_evaluation:
        r = arima_evaluation[col]
        arima_rmse = (r['forecast_error'] ** 2).mean() ** 0.5
        naive_rmse = (r['naive_error'] ** 2).mean() ** 0.5
        arima_theil = arima_rmse / naive_rmse
        arima_dir = (r['actual_direction'] == r['forecast_direction']).mean() * 100
        all_results.append({'pair': col.upper(), 'model': 'ARIMA', 'theil_u': arima_theil, 'direction_acc': arima_dir})
    
    # Chronos (from previous notebook)
    if col in chronos_results:
        all_results.append({
            'pair': col.upper(),
            'model': 'Chronos-Bolt',
            'theil_u': chronos_results[col]['theil_u'],
            'direction_acc': chronos_results[col]['direction_accuracy'] * 100
        })

comparison = pd.DataFrame(all_results)

for pair in ['USD_INR', 'USD_EUR', 'USD_GBP', 'USD_MXN', 'USD_PHP']:
    print(f"\n{pair}:")
    subset = comparison[comparison['pair'] == pair][['model', 'theil_u', 'direction_acc']]
    print(subset.round(4).to_string(index=False))

=== Classical Baselines vs Chronos ===


USD_INR:
       model  theil_u  direction_acc
         ETS   1.0046        53.3333
       ARIMA   0.9981        61.6667
Chronos-Bolt   1.0453        63.3333

USD_EUR:
       model  theil_u  direction_acc
         ETS   0.9950        61.6667
       ARIMA   1.0000        46.6667
Chronos-Bolt   1.0374        46.6667

USD_GBP:
       model  theil_u  direction_acc
         ETS   0.9951        58.3333
       ARIMA   0.9725        63.3333
Chronos-Bolt   1.0361        51.6667

USD_MXN:
       model  theil_u  direction_acc
         ETS   0.9945        60.0000
       ARIMA   0.9939        63.3333
Chronos-Bolt   0.9966        53.3333

USD_PHP:
       model  theil_u  direction_acc
         ETS   1.0036        55.0000
       ARIMA   1.0047        56.6667
Chronos-Bolt   1.0164        55.0000


In [5]:
print("=== Statistical Significance of Direction Accuracy ===\n")

for col in ['usd_inr', 'usd_eur', 'usd_gbp', 'usd_mxn', 'usd_php']:
    print(f"{col.upper()}:")
    
    for model_name, eval_dict in [('ETS', ets_evaluation), ('ARIMA', arima_evaluation)]:
        if col in eval_dict:
            r = eval_dict[col]
            n = len(r)
            correct = (r['actual_direction'] == r['forecast_direction']).sum()
            acc = correct / n * 100
            p_value = 1 - stats.binom.cdf(correct - 1, n, 0.5)
            sig = "✓" if p_value < 0.05 else "✗"
            
            print(f"  {model_name:<7} {correct}/{n} = {acc:.1f}%  p={p_value:.4f}  {sig}")
    print()

=== Statistical Significance of Direction Accuracy ===

USD_INR:
  ETS     32/60 = 53.3%  p=0.3494  ✗
  ARIMA   37/60 = 61.7%  p=0.0462  ✓

USD_EUR:
  ETS     37/60 = 61.7%  p=0.0462  ✓
  ARIMA   28/60 = 46.7%  p=0.7405  ✗

USD_GBP:
  ETS     35/60 = 58.3%  p=0.1225  ✗
  ARIMA   38/60 = 63.3%  p=0.0259  ✓

USD_MXN:
  ETS     36/60 = 60.0%  p=0.0775  ✗
  ARIMA   38/60 = 63.3%  p=0.0259  ✓

USD_PHP:
  ETS     33/60 = 55.0%  p=0.2595  ✗
  ARIMA   34/60 = 56.7%  p=0.1831  ✗



In [6]:
classical_output = {'ets': {}, 'arima': {}}

for col in ets_evaluation:
    r = ets_evaluation[col]
    rmse = (r['forecast_error'] ** 2).mean() ** 0.5
    naive_rmse = (r['naive_error'] ** 2).mean() ** 0.5
    classical_output['ets'][col] = {
        'model': 'auto_ets',
        'test_days': len(r),
        'rmse': float(rmse),
        'naive_rmse': float(naive_rmse),
        'theil_u': float(rmse / naive_rmse),
        'direction_accuracy': float((r['actual_direction'] == r['forecast_direction']).mean())
    }

for col in arima_evaluation:
    r = arima_evaluation[col]
    rmse = (r['forecast_error'] ** 2).mean() ** 0.5
    naive_rmse = (r['naive_error'] ** 2).mean() ** 0.5
    classical_output['arima'][col] = {
        'model': 'auto_arima',
        'test_days': len(r),
        'rmse': float(rmse),
        'naive_rmse': float(naive_rmse),
        'theil_u': float(rmse / naive_rmse),
        'direction_accuracy': float((r['actual_direction'] == r['forecast_direction']).mean())
    }

with open('classical_results.json', 'w') as f:
    json.dump(classical_output, f, indent=2)

print("Classical results saved to classical_results.json")
print("\nReady for notebook 09 — AutoGluon-TimeSeries")

Classical results saved to classical_results.json

Ready for notebook 09 — AutoGluon-TimeSeries


In [7]:
print("=== Direction Accuracy vs Majority Baseline ===\n")

majority_baselines = {
    'usd_inr': 56.4,
    'usd_eur': 53.8,
    'usd_gbp': 52.0,
    'usd_mxn': 54.2,  # majority is DOWN
    'usd_php': 56.4
}

for col in ['usd_inr', 'usd_eur', 'usd_gbp', 'usd_mxn', 'usd_php']:
    print(f"{col.upper()}: majority baseline = {majority_baselines[col]:.1f}%")
    
    for model_name, eval_dict in [('ETS', ets_evaluation), ('ARIMA', arima_evaluation)]:
        if col in eval_dict:
            r = eval_dict[col]
            n = len(r)
            correct = (r['actual_direction'] == r['forecast_direction']).sum()
            acc = correct / n * 100
            improvement = acc - majority_baselines[col]
            
            # Is the improvement over majority significant?
            p_value = 1 - stats.binom.cdf(correct - 1, n, majority_baselines[col] / 100)
            sig = "✓ beats majority" if p_value < 0.05 else "✗ does not beat majority"
            
            print(f"  {model_name:<7} {acc:.1f}%  ({improvement:+.1f}pp vs majority)  p={p_value:.4f}  {sig}")
    print()

=== Direction Accuracy vs Majority Baseline ===

USD_INR: majority baseline = 56.4%
  ETS     53.3%  (-3.1pp vs majority)  p=0.7298  ✗ does not beat majority
  ARIMA   61.7%  (+5.3pp vs majority)  p=0.2454  ✗ does not beat majority

USD_EUR: majority baseline = 53.8%
  ETS     61.7%  (+7.9pp vs majority)  p=0.1370  ✗ does not beat majority
  ARIMA   46.7%  (-7.1pp vs majority)  p=0.8919  ✗ does not beat majority

USD_GBP: majority baseline = 52.0%
  ETS     58.3%  (+6.3pp vs majority)  p=0.1971  ✗ does not beat majority
  ARIMA   63.3%  (+11.3pp vs majority)  p=0.0511  ✗ does not beat majority

USD_MXN: majority baseline = 54.2%
  ETS     60.0%  (+5.8pp vs majority)  p=0.2206  ✗ does not beat majority
  ARIMA   63.3%  (+9.1pp vs majority)  p=0.0978  ✗ does not beat majority

USD_PHP: majority baseline = 56.4%
  ETS     55.0%  (-1.4pp vs majority)  p=0.6381  ✗ does not beat majority
  ARIMA   56.7%  (+0.3pp vs majority)  p=0.5374  ✗ does not beat majority



In [8]:
print("=== What direction did each model actually predict? ===\n")

for col in ['usd_inr', 'usd_eur', 'usd_gbp', 'usd_mxn', 'usd_php']:
    print(f"{col.upper()}:")
    for model_name, eval_dict in [('ETS', ets_evaluation), ('ARIMA', arima_evaluation)]:
        if col in eval_dict:
            r = eval_dict[col]
            pred_up = (r['forecast_direction'] == 1).sum()
            actual_up = (r['actual_direction'] == 1).sum()
            total = len(r)
            print(f"  {model_name:<7} predicted UP {pred_up}/{total} = {pred_up/total*100:.0f}% (actual: {actual_up}/{total} = {actual_up/total*100:.0f}%)")
    print()

=== What direction did each model actually predict? ===

USD_INR:
  ETS     predicted UP 46/60 = 77% (actual: 38/60 = 63%)
  ARIMA   predicted UP 59/60 = 98% (actual: 38/60 = 63%)

USD_EUR:
  ETS     predicted UP 27/60 = 45% (actual: 32/60 = 53%)
  ARIMA   predicted UP 0/60 = 0% (actual: 32/60 = 53%)

USD_GBP:
  ETS     predicted UP 28/60 = 47% (actual: 33/60 = 55%)
  ARIMA   predicted UP 33/60 = 55% (actual: 33/60 = 55%)

USD_MXN:
  ETS     predicted UP 32/60 = 53% (actual: 28/60 = 47%)
  ARIMA   predicted UP 14/60 = 23% (actual: 28/60 = 47%)

USD_PHP:
  ETS     predicted UP 23/60 = 38% (actual: 36/60 = 60%)
  ARIMA   predicted UP 30/60 = 50% (actual: 36/60 = 60%)

